# Kazakh Language Understanding Benchmark — Gemini runs in Colab

Of five models only **Gemini** runs via API (temperature 0); the rest via web chat.
Colab gives open internet and keeps the key in Secrets.

1. Left panel 🔑 (**Secrets**) → add `GEMINI_API_KEY`, enable **Notebook access**.
2. Run cells top-to-bottom. Track A = fact-checking; Track B = idioms.

### 1. Clone repo + install + load key

In [ ]:
%cd /content
BRANCH = 'claude/kazakh-factcheck-benchmark-e76kd5'
REPO = 'github.com/Tim2190/kazakh-factcheck-benchmark.git'
!rm -rf /content/kazakh-factcheck-benchmark
!git clone --branch $BRANCH https://$REPO
%cd /content/kazakh-factcheck-benchmark
!pip install -q -r requirements.txt
import os
from google.colab import userdata
os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
print('OK')

### 2. Track A — fact-checking
`SOURCE`: `leg_text01` (Constitution), `news_text1` (news), or `msgtxt` (dialogue).

In [ ]:
SOURCE = 'msgtxt'
!python scripts/export_dataset.py
!python scripts/run_factcheck.py --model gemini --batch --source $SOURCE
!python scripts/check_grounding.py results/gemini_{SOURCE}_run.json
!python scripts/score.py results/gemini_{SOURCE}_run.json

### 3. Track B — idioms, phase 1
Runs all 30 idioms (bare, no context). Output is raw answers — grading is done
separately by a human annotator.

In [ ]:
!python scripts/run_idioms.py --model gemini --phase 1

### 4. Track B — idioms, phase 2 (optional, after phase-1 grading)
Only for idioms the model failed in phase 1. Put their ids in `FAILED_IDS`
(comma-separated) once grading is done, then run.

In [ ]:
FAILED_IDS = ''   # e.g. '2,8,15,18,21'
if FAILED_IDS.strip():
    !python scripts/run_idioms.py --model gemini --phase 2 --ids $FAILED_IDS
else:
    print('Set FAILED_IDS after phase-1 grading, then re-run this cell.')

### 5. Download results and send them to me

In [ ]:
from google.colab import files
import glob
for f in glob.glob('results/gemini_*_run.json') + glob.glob('idioms/results/gemini_*_raw.json'):
    print('download', f); files.download(f)